In [ ]:
!pip -q install rasterio geopandas shapely pyproj fiona albumentations segmentation-models-pytorch==0.3.3 timm==0.9.2

In [ ]:
import os

os.makedirs("/kaggle/working/dataset/images", exist_ok=True)
os.makedirs("/kaggle/working/dataset/labels", exist_ok=True)

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math

# --- 1. Setup Paths (Matching your screenshot) ---
img_path = "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_Training_dataSet_2/Training_dataSet_2/BADETUMNAR_450157_BANGAPAL_450155_CHHOTETUMAR_450149_MOFALNAR_450150_ORTHO.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Built_Up_Area_type.shp", "id": 1},
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Railway.shp", "id": 3},
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road.shp", "id": 4},
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Bridge.shp", "id": 5},
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    
    all_gdfs = []
    print("Loading shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
    
    master_gdf = gpd.pd.concat(all_gdfs, ignore_index=True)

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"Total possible tiles: {cols * rows}")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Create the Mask first to see if there is any data here
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            local_features = master_gdf[master_gdf.geometry.intersects(win_box)]
            
            # If no features in this tile, skip it (saves space!)
            if local_features.empty:
                continue
                
            mask_tile = np.zeros((tile_size, tile_size), dtype=np.uint8)
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(shapes, out_shape=(tile_size, tile_size), transform=tile_transform, all_touched=True)

            # 2. Read the Image tile
            image_tile = src.read(window=window, boundless=True, fill_value=0)
            
            # 3. Save to specific directories
            file_idx = f"{r}_{c}"
            # Saving as .npy for high precision (16-bit support)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile[:3])
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Saved {saved_count} tiles...")

print(f"--- Process Complete ---")
print(f"Total tiles saved in /kaggle/working/dataset: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math

# --- 1. Setup Paths (Matching your screenshot) ---
img_path = "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_Training_dataSet_3/Training_dataSet_3/NAGUL_450171_MADASE_450172_GHOTPAL_450137_ORTHO.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Built_Up_Area_type.shp", "id": 1},
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Railway.shp", "id": 3},
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road.shp", "id": 4},
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Bridge.shp", "id": 5},
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    
    all_gdfs = []
    print("Loading shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
    
    master_gdf = gpd.pd.concat(all_gdfs, ignore_index=True)

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"Total possible tiles: {cols * rows}")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Create the Mask first to see if there is any data here
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            local_features = master_gdf[master_gdf.geometry.intersects(win_box)]
            
            # If no features in this tile, skip it (saves space!)
            if local_features.empty:
                continue
                
            mask_tile = np.zeros((tile_size, tile_size), dtype=np.uint8)
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(shapes, out_shape=(tile_size, tile_size), transform=tile_transform, all_touched=True)

            # 2. Read the Image tile
            image_tile = src.read(window=window, boundless=True, fill_value=0)
            
            # 3. Save to specific directories
            file_idx = f"{r}_{c}"
            # Saving as .npy for high precision (16-bit support)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile[:3])
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Saved {saved_count} tiles...")

print(f"--- Process Complete ---")
print(f"Total tiles saved in /kaggle/working/dataset: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math
import pandas as pd # Needed for concatenation

# --- 1. Setup Paths ---
img_path = "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_Training_dataSet_3/Training_dataSet_3/NAGUL_450171_MADASE_450172_GHOTPAL_450137_ORTHO.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs - Updated with new files mapped to existing classes
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Built_Up_Area_type.shp", "id": 1},
    
    # Water Group (ID 2)
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Water_Line", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body_Line.shp", "id": 2},
    {"name": "Water_Point", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Waterbody_Point.shp", "id": 2},
    
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Railway.shp", "id": 3},
    
    # Road Group (ID 4)
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road.shp", "id": 4},
    {"name": "Road_Centerline", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road_Centre_Line.shp", "id": 4},
    
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Bridge.shp", "id": 5},
    
    # Utility Group (ID 6)
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility.shp", "id": 6},
    {"name": "Utility_Poly", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility_Poly.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    all_gdfs = []
    
    print("Loading and reprojecting shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if len(gdf) == 0: continue
            
            # Reproject to match Raster CRS
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
            print(f"  - Loaded {cfg['name']} ({len(gdf)} features)")
        else:
            print(f"  ! Missing file: {cfg['path']}")
    
    if not all_gdfs:
        raise ValueError("No valid shapefiles were loaded. Check your paths.")

    master_gdf = pd.concat(all_gdfs, ignore_index=True)
    # Create a spatial index to speed up intersections inside the loop
    sindex = master_gdf.sindex 

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"\nProcessing grid: {rows} rows x {cols} cols (Total potential: {cols * rows})")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            # Define window
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Spatial Filtering
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            
            # Use spatial index for fast filtering
            possible_matches_index = list(sindex.intersection(win_bounds))
            subset = master_gdf.iloc[possible_matches_index]
            local_features = subset[subset.intersects(win_box)]
            
            # Skip empty tiles (important for satellite imagery)
            if local_features.empty:
                continue
                
            # 2. Rasterize the mask
            # If multiple shapes overlap, the last one in the list wins.
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(
                shapes, 
                out_shape=(tile_size, tile_size), 
                transform=tile_transform, 
                all_touched=True, # Critical for lines and points!
                dtype=np.uint8
            )

            # 3. Read the Image tile
            # read(3) grabs first 3 bands. out_dtype=uint8 ensures consistent data types.
            image_tile = src.read([1, 2, 3], window=window, boundless=True, fill_value=0)
            
            # 4. Save
            file_idx = f"{r}_{c}"
            # Save Image (Bands, H, W) and Mask (H, W)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile.astype(np.uint8))
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Progress: Saved {saved_count} tiles...")

print(f"\n--- Process Complete ---")
print(f"Total tiles saved: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math
import pandas as pd # Needed for concatenation

# --- 1. Setup Paths ---
img_path = "/kaggle/input/datasets/golammasumbasar/rest-villages/SAMLUR_450163_SIYANAR_450164_KUTULNAR_450165_BINJAM_450166_JHODIYAWADAM_450167_ORTHO.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs - Updated with new files mapped to existing classes
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Built_Up_Area_type.shp", "id": 1},
    
    # Water Group (ID 2)
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Water_Line", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body_Line.shp", "id": 2},
    {"name": "Water_Point", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Waterbody_Point.shp", "id": 2},
    
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Railway.shp", "id": 3},
    
    # Road Group (ID 4)
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road.shp", "id": 4},
    {"name": "Road_Centerline", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road_Centre_Line.shp", "id": 4},
    
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Bridge.shp", "id": 5},
    
    # Utility Group (ID 6)
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility.shp", "id": 6},
    {"name": "Utility_Poly", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility_Poly.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    all_gdfs = []
    
    print("Loading and reprojecting shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if len(gdf) == 0: continue
            
            # Reproject to match Raster CRS
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
            print(f"  - Loaded {cfg['name']} ({len(gdf)} features)")
        else:
            print(f"  ! Missing file: {cfg['path']}")
    
    if not all_gdfs:
        raise ValueError("No valid shapefiles were loaded. Check your paths.")

    master_gdf = pd.concat(all_gdfs, ignore_index=True)
    # Create a spatial index to speed up intersections inside the loop
    sindex = master_gdf.sindex 

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"\nProcessing grid: {rows} rows x {cols} cols (Total potential: {cols * rows})")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            # Define window
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Spatial Filtering
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            
            # Use spatial index for fast filtering
            possible_matches_index = list(sindex.intersection(win_bounds))
            subset = master_gdf.iloc[possible_matches_index]
            local_features = subset[subset.intersects(win_box)]
            
            # Skip empty tiles (important for satellite imagery)
            if local_features.empty:
                continue
                
            # 2. Rasterize the mask
            # If multiple shapes overlap, the last one in the list wins.
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(
                shapes, 
                out_shape=(tile_size, tile_size), 
                transform=tile_transform, 
                all_touched=True, # Critical for lines and points!
                dtype=np.uint8
            )

            # 3. Read the Image tile
            # read(3) grabs first 3 bands. out_dtype=uint8 ensures consistent data types.
            image_tile = src.read([1, 2, 3], window=window, boundless=True, fill_value=0)
            
            # 4. Save
            file_idx = f"{r}_{c}"
            # Save Image (Bands, H, W) and Mask (H, W)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile.astype(np.uint8))
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Progress: Saved {saved_count} tiles...")

print(f"\n--- Process Complete ---")
print(f"Total tiles saved: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math
import pandas as pd # Needed for concatenation

# --- 1. Setup Paths ---
img_path = "/kaggle/input/datasets/golammasumbasar/rest-villages/37458_fattu_bhila_ortho_3857.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs - Updated with new files mapped to existing classes
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Built_Up_Area_typ.shp", "id": 1},
    
    # Water Group (ID 2)
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Water_Line", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body_Line.shp", "id": 2},
    {"name": "Water_Point", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Waterbody_Point.shp", "id": 2},
    
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Railway.shp", "id": 3},
    
    # Road Group (ID 4)
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road.shp", "id": 4},
    {"name": "Road_Centerline", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road_Centre_Line.shp", "id": 4},
    
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Bridge.shp", "id": 5},
    
    # Utility Group (ID 6)
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility.shp", "id": 6},
    {"name": "Utility_Poly", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility_Poly_.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    all_gdfs = []
    
    print("Loading and reprojecting shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if len(gdf) == 0: continue
            
            # Reproject to match Raster CRS
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
            print(f"  - Loaded {cfg['name']} ({len(gdf)} features)")
        else:
            print(f"  ! Missing file: {cfg['path']}")
    
    if not all_gdfs:
        raise ValueError("No valid shapefiles were loaded. Check your paths.")

    master_gdf = pd.concat(all_gdfs, ignore_index=True)
    # Create a spatial index to speed up intersections inside the loop
    sindex = master_gdf.sindex 

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"\nProcessing grid: {rows} rows x {cols} cols (Total potential: {cols * rows})")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            # Define window
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Spatial Filtering
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            
            # Use spatial index for fast filtering
            possible_matches_index = list(sindex.intersection(win_bounds))
            subset = master_gdf.iloc[possible_matches_index]
            local_features = subset[subset.intersects(win_box)]
            
            # Skip empty tiles (important for satellite imagery)
            if local_features.empty:
                continue
                
            # 2. Rasterize the mask
            # If multiple shapes overlap, the last one in the list wins.
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(
                shapes, 
                out_shape=(tile_size, tile_size), 
                transform=tile_transform, 
                all_touched=True, # Critical for lines and points!
                dtype=np.uint8
            )

            # 3. Read the Image tile
            # read(3) grabs first 3 bands. out_dtype=uint8 ensures consistent data types.
            image_tile = src.read([1, 2, 3], window=window, boundless=True, fill_value=0)
            
            # 4. Save
            file_idx = f"{r}_{c}"
            # Save Image (Bands, H, W) and Mask (H, W)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile.astype(np.uint8))
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Progress: Saved {saved_count} tiles...")

print(f"\n--- Process Complete ---")
print(f"Total tiles saved: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math
import pandas as pd # Needed for concatenation

# --- 1. Setup Paths ---
img_path = "/kaggle/input/datasets/golammasumbasar/rest-villages/37774_bagga ortho_3857.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs - Updated with new files mapped to existing classes
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Built_Up_Area_typ.shp", "id": 1},
    
    # Water Group (ID 2)
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Water_Line", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body_Line.shp", "id": 2},
    {"name": "Water_Point", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Waterbody_Point.shp", "id": 2},
    
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Railway.shp", "id": 3},
    
    # Road Group (ID 4)
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road.shp", "id": 4},
    {"name": "Road_Centerline", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road_Centre_Line.shp", "id": 4},
    
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Bridge.shp", "id": 5},
    
    # Utility Group (ID 6)
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility.shp", "id": 6},
    {"name": "Utility_Poly", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility_Poly_.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    all_gdfs = []
    
    print("Loading and reprojecting shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if len(gdf) == 0: continue
            
            # Reproject to match Raster CRS
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
            print(f"  - Loaded {cfg['name']} ({len(gdf)} features)")
        else:
            print(f"  ! Missing file: {cfg['path']}")
    
    if not all_gdfs:
        raise ValueError("No valid shapefiles were loaded. Check your paths.")

    master_gdf = pd.concat(all_gdfs, ignore_index=True)
    # Create a spatial index to speed up intersections inside the loop
    sindex = master_gdf.sindex 

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"\nProcessing grid: {rows} rows x {cols} cols (Total potential: {cols * rows})")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            # Define window
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Spatial Filtering
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            
            # Use spatial index for fast filtering
            possible_matches_index = list(sindex.intersection(win_bounds))
            subset = master_gdf.iloc[possible_matches_index]
            local_features = subset[subset.intersects(win_box)]
            
            # Skip empty tiles (important for satellite imagery)
            if local_features.empty:
                continue
                
            # 2. Rasterize the mask
            # If multiple shapes overlap, the last one in the list wins.
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(
                shapes, 
                out_shape=(tile_size, tile_size), 
                transform=tile_transform, 
                all_touched=True, # Critical for lines and points!
                dtype=np.uint8
            )

            # 3. Read the Image tile
            # read(3) grabs first 3 bands. out_dtype=uint8 ensures consistent data types.
            image_tile = src.read([1, 2, 3], window=window, boundless=True, fill_value=0)
            
            # 4. Save
            file_idx = f"{r}_{c}"
            # Save Image (Bands, H, W) and Mask (H, W)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile.astype(np.uint8))
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Progress: Saved {saved_count} tiles...")

print(f"\n--- Process Complete ---")
print(f"Total tiles saved: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math
import pandas as pd # Needed for concatenation

# --- 1. Setup Paths ---
img_path = "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/TIMMOWAL_37695_ORI.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs - Updated with new files mapped to existing classes
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Built_Up_Area_typ.shp", "id": 1},
    
    # Water Group (ID 2)
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Water_Line", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Water_Body_Line.shp", "id": 2},
    {"name": "Water_Point", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Waterbody_Point.shp", "id": 2},
    
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Railway.shp", "id": 3},
    
    # Road Group (ID 4)
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road.shp", "id": 4},
    {"name": "Road_Centerline", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Road_Centre_Line.shp", "id": 4},
    
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Bridge.shp", "id": 5},
    
    # Utility Group (ID 6)
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility.shp", "id": 6},
    {"name": "Utility_Poly", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/PB_training_dataSet_shp_file/PB_training_dataSet_shp_file/shp-file/Utility_Poly_.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    all_gdfs = []
    
    print("Loading and reprojecting shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if len(gdf) == 0: continue
            
            # Reproject to match Raster CRS
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
            print(f"  - Loaded {cfg['name']} ({len(gdf)} features)")
        else:
            print(f"  ! Missing file: {cfg['path']}")
    
    if not all_gdfs:
        raise ValueError("No valid shapefiles were loaded. Check your paths.")

    master_gdf = pd.concat(all_gdfs, ignore_index=True)
    # Create a spatial index to speed up intersections inside the loop
    sindex = master_gdf.sindex 

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"\nProcessing grid: {rows} rows x {cols} cols (Total potential: {cols * rows})")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            # Define window
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Spatial Filtering
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            
            # Use spatial index for fast filtering
            possible_matches_index = list(sindex.intersection(win_bounds))
            subset = master_gdf.iloc[possible_matches_index]
            local_features = subset[subset.intersects(win_box)]
            
            # Skip empty tiles (important for satellite imagery)
            if local_features.empty:
                continue
                
            # 2. Rasterize the mask
            # If multiple shapes overlap, the last one in the list wins.
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(
                shapes, 
                out_shape=(tile_size, tile_size), 
                transform=tile_transform, 
                all_touched=True, # Critical for lines and points!
                dtype=np.uint8
            )

            # 3. Read the Image tile
            # read(3) grabs first 3 bands. out_dtype=uint8 ensures consistent data types.
            image_tile = src.read([1, 2, 3], window=window, boundless=True, fill_value=0)
            
            # 4. Save
            file_idx = f"{r}_{c}"
            # Save Image (Bands, H, W) and Mask (H, W)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile.astype(np.uint8))
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Progress: Saved {saved_count} tiles...")

print(f"\n--- Process Complete ---")
print(f"Total tiles saved: {saved_count}")

In [ ]:
import os
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import geopandas as gpd
import numpy as np
from shapely.geometry import box
import math

# --- 1. Setup Paths (Matching your screenshot) ---
img_path = "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_Training_dataSet_3/Training_dataSet_3/MURDANDA_450879_AWAPALLI_CHINTAKONTA_ORTHO.tif"
base_dir = "/kaggle/working/dataset"
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

# Define shapefiles and IDs
shp_configs = [
    {"name": "Built_Up", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Built_Up_Area_type.shp", "id": 1},
    {"name": "Water", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Water_Body.shp", "id": 2},
    {"name": "Railway", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Railway.shp", "id": 3},
    {"name": "Road", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Road.shp", "id": 4},
    {"name": "Bridge", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Bridge.shp", "id": 5},
    {"name": "Utility", "path": "/kaggle/input/datasets/golammasumbasar/cg-training-datasets/CG_shp-file/shp-file/Utility.shp", "id": 6}
]

tile_size = 256

# --- 2. Load and Prepare All Shapes ---
with rasterio.open(img_path) as src:
    raster_crs = src.crs
    
    all_gdfs = []
    print("Loading shapefiles...")
    for cfg in shp_configs:
        if os.path.exists(cfg['path']):
            gdf = gpd.read_file(cfg['path'])
            if gdf.crs != raster_crs:
                gdf = gdf.to_crs(raster_crs)
            gdf['class_id'] = cfg['id']
            all_gdfs.append(gdf[['geometry', 'class_id']])
    
    master_gdf = gpd.pd.concat(all_gdfs, ignore_index=True)

    # --- 3. Grid Tiling Logic ---
    width, height = src.width, src.height
    cols = math.ceil(width / tile_size)
    rows = math.ceil(height / tile_size)

    print(f"Total possible tiles: {cols * rows}")
    saved_count = 0

    for r in range(rows):
        for c in range(cols):
            x = c * tile_size
            y = r * tile_size
            
            window = Window(x, y, tile_size, tile_size)
            tile_transform = src.window_transform(window)
            
            # 1. Create the Mask first to see if there is any data here
            win_bounds = src.window_bounds(window)
            win_box = box(*win_bounds)
            local_features = master_gdf[master_gdf.geometry.intersects(win_box)]
            
            # If no features in this tile, skip it (saves space!)
            if local_features.empty:
                continue
                
            mask_tile = np.zeros((tile_size, tile_size), dtype=np.uint8)
            shapes = [(geom, val) for geom, val in zip(local_features.geometry, local_features.class_id)]
            mask_tile = rasterize(shapes, out_shape=(tile_size, tile_size), transform=tile_transform, all_touched=True)

            # 2. Read the Image tile
            image_tile = src.read(window=window, boundless=True, fill_value=0)
            
            # 3. Save to specific directories
            file_idx = f"{r}_{c}"
            # Saving as .npy for high precision (16-bit support)
            np.save(os.path.join(images_dir, f"tile_{file_idx}.npy"), image_tile[:3])
            np.save(os.path.join(labels_dir, f"mask_{file_idx}.npy"), mask_tile)

            saved_count += 1
            if saved_count % 100 == 0:
                print(f"Saved {saved_count} tiles...")

print(f"--- Process Complete ---")
print(f"Total tiles saved in /kaggle/working/dataset: {saved_count}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Pick a random saved tile index from your 'labels' folder
label_files = os.listdir("/kaggle/working/dataset/labels")
sample_file = label_files[1001] # Just pick the 10th one
image_file = sample_file.replace("mask", "tile")

# Load them back
img = np.load(f"/kaggle/working/dataset/images/{image_file}")
mask = np.load(f"/kaggle/working/dataset/labels/{sample_file}")

# Display
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img[:3].transpose(1, 2, 0).astype(float) / img.max())
ax[0].set_title("Saved Image Tile")
ax[1].imshow(mask, cmap='terrain')
ax[1].set_title("Saved Mask Tile")
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import numpy as np

class SatelliteDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        # Filtering to ensure we only load files that exist in both folders
        self.images = sorted(os.listdir(image_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = img_name.replace("tile_", "mask_") # Adjust based on your naming
        
        img = np.load(os.path.join(self.image_dir, img_name))
        mask = np.load(os.path.join(self.mask_dir, mask_name))

        # Reorder from (C, H, W) to (H, W, C) for Albumentations
        img = np.transpose(img, (1, 2, 0))

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        return img, mask.long()

# --- 2. Data Augmentation (Crucial for Satellite Imagery) ---
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
# Run this before training to see class distribution across all masks
import os, numpy as np
from pathlib import Path

msk_dir = "/kaggle/working/dataset/labels"
counts  = np.zeros(7, dtype=np.int64)

for f in os.listdir(msk_dir):
    if not f.endswith('.npy'):
        continue
    m = np.load(os.path.join(msk_dir, f))
    for c in range(7):
        counts[c] += (m == c).sum()

names = ["background","built_up","water","railway","road","bridge","utility"]
total = counts.sum()
print("Class distribution across all tiles:")
for i, (n, c) in enumerate(zip(names, counts)):
    print(f"  {i} {n:12s}: {c:10,}  ({100*c/total:.2f}%)")

In [ ]:
# ── Remap masks: drop railway, shift IDs down ─────────────────
# OLD: 0=bg 1=built 2=water 3=railway 4=road 5=bridge 6=utility
# NEW: 0=bg 1=built 2=water 3=road   4=bridge 5=utility
# Railway had 0 pixels — safe to remove

import os, numpy as np
from tqdm.auto import tqdm

MASK_DIR = "/kaggle/working/dataset/labels"

REMAP = {
    0: 0,   # background → background
    1: 1,   # built_up   → built_up
    2: 2,   # water      → water
    3: 0,   # railway    → background (zero pixels anyway)
    4: 3,   # road       → road
    5: 4,   # bridge     → bridge
    6: 5,   # utility    → utility
}

files = [f for f in os.listdir(MASK_DIR) if f.endswith('.npy')]
print(f"Remapping {len(files)} mask files...")

for fname in tqdm(files):
    path = os.path.join(MASK_DIR, fname)
    mask = np.load(path)
    new  = np.zeros_like(mask)
    for old_id, new_id in REMAP.items():
        new[mask == old_id] = new_id
    np.save(path, new)

print("✓ Done. New IDs: 0=bg 1=built 2=water 3=road 4=bridge 5=utility")

# Verify
sample = np.load(os.path.join(MASK_DIR, files[0]))
print(f"Unique values in first mask: {np.unique(sample)}")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# This must work without error before anything else
x = torch.tensor([1.0, 2.0]).to("cuda")
print("CUDA OK:", x)

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import (
    DataLoader, Dataset, WeightedRandomSampler, random_split
)
from tqdm.auto import tqdm

# ── CONFIG ────────────────────────────────────────────────────
IMAGE_DIR   = "/kaggle/working/dataset/images"
MASK_DIR    = "/kaggle/working/dataset/labels"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 4      # drop to 2 if OOM
NUM_CLASSES = 6      # 0=bg 1=built 2=water 3=road 4=bridge 5=utility
MAX_EPOCHS  = 25     # ceiling — early stop fires well before this
PATIENCE    = 4      # stop if no val improvement for 4 epochs

CLASS_WEIGHTS = torch.tensor([
    0.5,    # 0 background   55%
    1.5,    # 1 built_up     22%
    2.0,    # 2 water        12%
    3.0,    # 3 road          9%
    15.0,   # 4 bridge       0.01%
    12.0,   # 5 utility      0.04%
], dtype=torch.float32).to(DEVICE)

print(f"Device      : {DEVICE}")
print(f"NUM_CLASSES : {NUM_CLASSES}")
print(f"Weights     : {CLASS_WEIGHTS.tolist()}")


# ── DATASET ───────────────────────────────────────────────────
class TileDataset(Dataset):
    def __init__(self, img_dir, msk_dir, preprocess_fn=None):
        self.img_dir    = img_dir
        self.msk_dir    = msk_dir
        self.preprocess = preprocess_fn
        self.files      = sorted([
            f for f in os.listdir(img_dir) if f.endswith('.npy')
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        # Image — saved as (3,256,256) uint8
        img = np.load(os.path.join(self.img_dir, fname))
        if img.ndim == 3 and img.shape[0] == 3:
            img = img.transpose(1, 2, 0)        # CHW → HWC
        img = img[:, :, :3]
        if img.dtype != np.uint8:
            img = img.clip(0, 255).astype(np.uint8)

        # Mask
        mname = fname.replace("tile_", "mask_")
        mask  = np.load(os.path.join(self.msk_dir, mname)).astype(np.int64)
        mask  = np.clip(mask, 0, NUM_CLASSES - 1)

        if self.preprocess:
            img = self.preprocess(img)          # → float32 HWC normalised

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        msk_t = torch.from_numpy(mask).long()
        return img_t, msk_t


# ── OVERSAMPLER ───────────────────────────────────────────────
def make_sampler(dataset):
    """
    Tiles containing bridge or utility get sampled 15-20x more.
    Applied to training set only.
    """
    TILE_WEIGHT = {0:1.0, 1:1.0, 2:1.2, 3:1.5, 4:20.0, 5:15.0}
    weights = []
    for fname in dataset.files:
        mpath = os.path.join(dataset.msk_dir,
                             fname.replace("tile_", "mask_"))
        mask  = np.load(mpath)
        w     = max(TILE_WEIGHT.get(int(c), 1.0) for c in np.unique(mask))
        weights.append(w)
    weights = torch.tensor(weights, dtype=torch.float32)
    return WeightedRandomSampler(
        weights, num_samples=len(weights), replacement=True
    )


# ── LOSS ──────────────────────────────────────────────────────
dice_loss = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
ce_loss   = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

def criterion(pred, target):
    return 0.5 * dice_loss(pred, target) + 0.5 * ce_loss(pred, target)


# ── SUBSET HELPER ─────────────────────────────────────────────
class IndexedSubset(Dataset):
    def __init__(self, img_dir, msk_dir, indices, preprocess_fn):
        self.base    = TileDataset(img_dir, msk_dir, preprocess_fn)
        self.indices = list(indices)

    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]

    @property
    def files(self):
        return [self.base.files[i] for i in self.indices]

    @property
    def msk_dir(self): return self.base.msk_dir


# ── SHARED TRAIN/VAL SPLIT ────────────────────────────────────
_base = TileDataset(IMAGE_DIR, MASK_DIR)
n_total = len(_base)
n_train = int(0.85 * n_total)
n_val   = n_total - n_train
g       = torch.Generator().manual_seed(42)
_split  = random_split(range(n_total), [n_train, n_val], generator=g)
TR_IDX, VL_IDX = list(_split[0]), list(_split[1])
print(f"\nTrain tiles : {n_train}")
print(f"Val tiles   : {n_val}")


def make_loaders(preprocess_fn):
    tr_ds = IndexedSubset(IMAGE_DIR, MASK_DIR, TR_IDX, preprocess_fn)
    vl_ds = IndexedSubset(IMAGE_DIR, MASK_DIR, VL_IDX, preprocess_fn)

    print("  Building oversampler...")
    sampler = make_sampler(tr_ds)

    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE,
                       sampler=sampler,
                       num_workers=2, pin_memory=True)
    vl_ld = DataLoader(vl_ds, batch_size=BATCH_SIZE,
                       shuffle=False,
                       num_workers=2, pin_memory=True)
    return tr_ld, vl_ld


# ── TRAIN FUNCTION WITH EARLY STOPPING ───────────────────────
def train_model(model, name, tr_loader, vl_loader,
                max_epochs=MAX_EPOCHS, patience=PATIENCE):
    opt   = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    path  = f"/kaggle/working/{name}.pth"

    best_val   = float('inf')
    no_improve = 0
    best_epoch = 0

    print(f"\nStarting {name}  (max {max_epochs} epochs, patience {patience})")
    print("-" * 55)

    for ep in range(1, max_epochs + 1):
        # Train
        model.train()
        tl = 0.0
        for imgs, msks in tqdm(tr_loader, desc=f"Train ep{ep}"):
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(imgs), msks)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item()

        # Validate
        model.eval()
        vl = 0.0
        with torch.no_grad():
            for imgs, msks in vl_loader:
                imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
                vl += criterion(model(imgs), msks).item()

        avg_t = tl / len(tr_loader)
        avg_v = vl / len(vl_loader)

        if avg_v < best_val:
            best_val   = avg_v
            best_epoch = ep
            no_improve = 0
            torch.save(model.state_dict(), path)
            tag = "  ✓ best saved"
        else:
            no_improve += 1
            tag = f"  (no improve {no_improve}/{patience})"

        print(f"  ep {ep:02d}  train {avg_t:.4f}  val {avg_v:.4f}{tag}")
        sched.step()

        if no_improve >= patience:
            print(f"\n  ⬛ Early stop. Best epoch: {best_epoch}  val: {best_val:.4f}")
            break

    print(f"\n✓ {name} done.  Best val: {best_val:.4f}  →  {path}")
    return path


# ── TRAIN MODEL A: UNet++ EfficientNet-B5 ────────────────────
print("\n" + "="*55)
print("MODEL A — UNet++  EfficientNet-B5  (weight 0.4)")
print("="*55)

prep_b5 = smp.encoders.get_preprocessing_fn("efficientnet-b5", "imagenet")
model_a = smp.UnetPlusPlus(
    encoder_name    = "efficientnet-b5",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = NUM_CLASSES,
).to(DEVICE)

tr_a, vl_a = make_loaders(prep_b5)
path_a = train_model(model_a, "model_A_unetpp_b5", tr_a, vl_a)

del model_a, tr_a, vl_a
torch.cuda.empty_cache()


# ── TRAIN MODEL B: DeepLabV3+ ResNet-101 ─────────────────────
print("\n" + "="*55)
print("MODEL B — DeepLabV3+  ResNet-101  (weight 0.6)")
print("="*55)

prep_r101 = smp.encoders.get_preprocessing_fn("resnet101", "imagenet")
model_b   = smp.DeepLabV3Plus(
    encoder_name    = "resnet101",
    encoder_weights = "imagenet",
    in_channels     = 3,
    classes         = NUM_CLASSES,
).to(DEVICE)

tr_b, vl_b = make_loaders(prep_r101)
path_b = train_model(model_b, "model_B_deeplabv3p_r101", tr_b, vl_b)

del model_b, tr_b, vl_b
torch.cuda.empty_cache()

print("\n" + "="*55)
print("BOTH MODELS TRAINED")
print(f"  A → {path_a}")
print(f"  B → {path_b}")
print("="*55)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
from scipy.optimize import minimize

# 1. Configuration (Ensure these match your previous setup)
NUM_CLASSES = 6
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["background","built_up","water","road","bridge","utility"]

# 2. Load Models
model_a = smp.UnetPlusPlus(encoder_name="efficientnet-b5", encoder_weights=None, classes=NUM_CLASSES).to(DEVICE)
model_a.load_state_dict(torch.load("/kaggle/working/model_A_unetpp_b5.pth"))
model_a.eval()

model_b = smp.DeepLabV3Plus(encoder_name="resnet101", encoder_weights=None, classes=NUM_CLASSES).to(DEVICE)
model_b.load_state_dict(torch.load("/kaggle/working/model_B_deeplabv3p_r101.pth"))
model_b.eval()

# 3. Setup DataLoaders (This defines vl_a and vl_b)
prep_b5   = smp.encoders.get_preprocessing_fn("efficientnet-b5", "imagenet")
prep_r101 = smp.encoders.get_preprocessing_fn("resnet101", "imagenet")

# Note: Ensure IMAGE_DIR, MASK_DIR, and VL_IDX are defined in your notebook
vl_ds_a = IndexedSubset(IMAGE_DIR, MASK_DIR, VL_IDX, prep_b5)
vl_ds_b = IndexedSubset(IMAGE_DIR, MASK_DIR, VL_IDX, prep_r101)

vl_a = DataLoader(vl_ds_a, batch_size=4, shuffle=False, num_workers=2)
vl_b = DataLoader(vl_ds_b, batch_size=4, shuffle=False, num_workers=2)

# 4. Extract Probabilities
all_probs_a, all_probs_b, all_masks = [], [], []

print("Extracting probabilities (this may take a few minutes)...")
with torch.no_grad():
    for (imgs_a, msks), (imgs_b, _) in zip(vl_a, vl_b):
        p_a = F.softmax(model_a(imgs_a.to(DEVICE)), dim=1).cpu().numpy()
        p_b = F.softmax(model_b(imgs_b.to(DEVICE)), dim=1).cpu().numpy()
        all_probs_a.append(p_a)
        all_probs_b.append(p_b)
        all_masks.append(msks.numpy())

# Convert to large numpy arrays for the optimizer
all_probs_a = np.concatenate(all_probs_a)
all_probs_b = np.concatenate(all_probs_b)
all_masks   = np.concatenate(all_masks)

# 5. Optimization Function
def objective(weights):
    w_a, w_b = weights
    blended = (w_a * all_probs_a) + (w_b * all_probs_b)
    preds = blended.argmax(axis=1)
    
    ious = []
    for c in range(NUM_CLASSES):
        it = np.logical_and(preds == c, all_masks == c).sum()
        un = np.logical_or(preds == c, all_masks == c).sum()
        if un > 0: ious.append(it / un)
    return -np.mean(ious) # Maximize Mean IoU

# 6. Find Best Weights
cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
res = minimize(objective, [0.5, 0.5], method='SLSQP', bounds=[(0,1),(0,1)], constraints=cons)

best_wa, best_wb = res.x
print(f"\nOptimal Weights -> Model A: {best_wa:.4f}, Model B: {best_wb:.4f}")
print(f"Optimized mIoU: {-res.fun:.4f}")

In [ ]:
import numpy as np
from scipy.optimize import minimize

def optimize_per_class(probs_a, probs_b, masks, num_classes):
    best_weights_a = np.zeros(num_classes)
    best_weights_b = np.zeros(num_classes)
    class_ious = []

    print(f"{'Class':<12} | {'Weight A':<10} | {'Weight B':<10} | {'IoU':<10}")
    print("-" * 50)

    for c in range(num_classes):
        # Objective function for a single class
        def class_objective(w):
            w_a, w_b = w
            # Combine probabilities for this specific class only
            combined_prob_c = (w_a * probs_a[:, c, :, :]) + (w_b * probs_b[:, c, :, :])
            
            # Since we are doing this per-class, we need to compare against 
            # other classes to get a "hard" prediction, but a shortcut is 
            # to maximize the correlation with the ground truth mask for class C
            preds_c = combined_prob_c > 0.5 # Thresholding for optimization
            true_c = (masks == c)
            
            intersection = np.logical_and(preds_c, true_c).sum()
            union = np.logical_or(preds_c, true_c).sum()
            
            return -(intersection / (union + 1e-7))

        cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
        res = minimize(class_objective, [0.5, 0.5], method='SLSQP', 
                       bounds=[(0, 1), (0, 1)], constraints=cons)
        
        best_weights_a[c], best_weights_b[c] = res.x
        class_ious.append(-res.fun)
        
        print(f"{CLASS_NAMES[c]:<12} | {res.x[0]:.4f}     | {res.x[1]:.4f}     | {(-res.fun):.4f}")

    print("-" * 50)
    print(f"Estimated Potential mIoU: {np.mean(class_ious):.4f}")
    return best_weights_a, best_weights_b

# Run the optimizer
best_wa_list, best_wb_list = optimize_per_class(all_probs_a, all_probs_b, all_masks, NUM_CLASSES)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

# ── CONFIG ────────────────────────────────────────────────────
IMAGE_DIR   = "/kaggle/working/dataset/images"
MASK_DIR    = "/kaggle/working/dataset/labels"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 8  # Increased for better gradient stability
NUM_CLASSES = 6
MAX_EPOCHS  = 50 
PATIENCE    = 5  # Increased patience slightly for late-stage gains

# Focused Weights: Slightly less aggressive than before to prevent over-fitting on noise
CLASS_WEIGHTS = torch.tensor([1.0, 1.2, 1.2, 3.0, 4.0, 6.0]).to(DEVICE)

# ── DATASET ───────────────────────────────────────────────────
class TileDataset(Dataset):
    def __init__(self, img_dir, msk_dir, preprocess_fn=None):
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        self.preprocess = preprocess_fn
        self.files = sorted([f for f in os.listdir(img_dir) if f.endswith('.npy')])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = np.load(os.path.join(self.img_dir, fname))
        if img.ndim == 3 and img.shape[0] == 3: img = img.transpose(1, 2, 0)
        
        mname = fname.replace("tile_", "mask_")
        mask = np.load(os.path.join(self.msk_dir, mname)).astype(np.int64)
        
        if self.preprocess:
            # Note: For max accuracy, consider adding Albumentations here later
            img = self.preprocess(img)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        msk_t = torch.from_numpy(mask).long()
        return img_t, msk_t

# ── METRIC CALCULATION ────────────────────────────────────────
def get_miou(preds, targets):
    ious = []
    preds = torch.argmax(preds, dim=1)
    for c in range(NUM_CLASSES):
        inter = ((preds == c) & (targets == c)).sum().item()
        union = ((preds == c) | (targets == c)).sum().item()
        if union > 0:
            ious.append(inter / union)
    return np.mean(ious) if ious else 0

# ── LOSS ──────────────────────────────────────────────────────
# Dice + Weighted CrossEntropy for balanced accuracy
dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss   = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

def criterion(pred, target):
    return 0.5 * dice_loss(pred, target) + 0.5 * ce_loss(pred, target)

# ── PREPARE DATA ──────────────────────────────────────────────
prep_b5 = smp.encoders.get_preprocessing_fn("efficientnet-b5", "imagenet")
full_ds = TileDataset(IMAGE_DIR, MASK_DIR, prep_b5)

n_train = int(0.85 * len(full_ds))
n_val = len(full_ds) - n_train
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── MODEL INITIALIZATION ──────────────────────────────────────
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# ── TRAINING LOOP ─────────────────────────────────────────────
best_miou = 0.0
no_improve = 0

print(f"Starting Training for Model A (U-Net++) on {DEVICE}")
print("-" * 60)

for ep in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    
    for imgs, msks in tqdm(train_loader, desc=f"Epoch {ep}"):
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer.zero_grad()
        output = model(imgs)
        loss = criterion(output, msks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    val_miou = 0.0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            output = model(imgs)
            val_loss += criterion(output, msks).item()
            val_miou += get_miou(output, msks)

    avg_v_loss = val_loss / len(val_loader)
    avg_v_miou = val_miou / len(val_loader)
    
    print(f"Ep {ep} | Val Loss: {avg_v_loss:.4f} | Val mIoU: {avg_v_miou:.4f}")

    scheduler.step(avg_v_miou)

    if avg_v_miou > best_miou:
        best_miou = avg_v_miou
        torch.save(model.state_dict(), "/kaggle/working/best_unetpp_b5_accuracy.pth")
        print(f"  ✅ mIoU Improved! Saved Model.")
        no_improve = 0
    else:
        no_improve += 1
        print(f"  ❌ No improvement ({no_improve}/{PATIENCE})")

    if no_improve >= PATIENCE:
        print(f"\nEarly Stopping triggered. Best Val mIoU: {best_miou:.4f}")
        break